# InstaNovo evaluation — Ecoli_EV_2 (held-out test)

Runs `instanovo transformer predict --evaluation` against
`Ecoli_EV_2.instanovo.annotated.mgf` with **both** the pretrained and the
fine-tuned InstaNovo checkpoints. Saves both prediction CSVs to Drive,
and prints peptide-level exact-match rates side-by-side as a quick sanity
check.

**Inputs (must exist on Drive):**
- `MyDrive/DL-Project/data_mgf_annotated/ecoli/Ecoli_EV_2.instanovo.annotated.mgf`  (UNIMOD notation)
- `MyDrive/DL-Project/bin/instanovo/instanovo-v1.2.0.ckpt`             (pretrained)
- `MyDrive/DL-Project/model_finetune/instanovo/model_best.ckpt`       (fine-tuned, output of `instanovo_colab_finetune.ipynb`)

**Outputs (written to Drive):**
- `MyDrive/DL-Project/result_finetune/instanovo/Ecoli_EV_2.pretrained.csv`
- `MyDrive/DL-Project/result_finetune/instanovo/Ecoli_EV_2.finetuned.csv`

**Note on notation:** the annotated MGF is generated by
`annotate_mgf.py --notation unimod`, so SEQ= lines are already in
InstaNovo's native UNIMOD-bracket form (`M[UNIMOD:35]`, `C[UNIMOD:4]`,
...). No translation required at predict time.


In [1]:
!nvidia-smi


Sat May  2 02:23:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             50W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install dependencies


In [2]:
try:
  import instanovo
except ImportError:
  !pip install "instanovo[cu126]>=1.2.2" pyopenms-viz
  print('Installation complete. Restarting runtime to apply changes...')
  import os
  os.kill(os.getpid(), 9)


## Sync inputs from Drive


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/data_mgf_annotated/ecoli', exist_ok=True)
os.makedirs('/content/bin/instanovo', exist_ok=True)
os.makedirs('/content/model_finetune/instanovo', exist_ok=True)
os.makedirs('/content/result_finetune/instanovo', exist_ok=True)

# Annotated test MGF (UNIMOD notation — produced by annotate_mgf.py --notation unimod)
!cp /content/drive/MyDrive/DL-Project/data_mgf_annotated/ecoli/Ecoli_EV_2.instanovo.annotated.mgf /content/data_mgf_annotated/ecoli/

# Pretrained ckpt
!cp /content/drive/MyDrive/DL-Project/bin/instanovo/instanovo-v1.2.0.ckpt /content/bin/instanovo/instanovo-v1.2.0.ckpt

# Fine-tuned ckpt (output of instanovo_colab_finetune.ipynb)
!cp /content/drive/MyDrive/DL-Project/model_finetune/instanovo/model_best.ckpt /content/model_finetune/instanovo/model_best.ckpt

print('--- inputs ---')
!ls -lh /content/data_mgf_annotated/ecoli/
!ls -lh /content/bin/instanovo/
!ls -lh /content/model_finetune/instanovo/

# Sanity check: SEQ= lines should already be in UNIMOD form on disk.
print('\n--- first SEQ= entries (expect UNIMOD bracket form) ---')
!grep -m3 '^SEQ=' /content/data_mgf_annotated/ecoli/Ecoli_EV_2.instanovo.annotated.mgf


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- inputs ---
total 66M
-rw------- 1 root root  27M May  2 02:09 Ecoli_EV_1.instanovo.train.mgf
-rw------- 1 root root 5.0M May  2 02:09 Ecoli_EV_1.instanovo.val.mgf
-rw------- 1 root root  34M May  2 02:24 Ecoli_EV_2.instanovo.annotated.mgf
total 362M
-rw------- 1 root root 362M May  2 02:24 instanovo-v1.2.0.ckpt
total 724M
drwxr-xr-x 4 root root 4.0K May  2 02:13 accelerator_state
-rw-r--r-- 1 root root 362M May  2 02:24 model_best.ckpt
-rw-r--r-- 1 root root 362M May  2 02:13 model_latest.ckpt

--- first SEQ= entries (expect UNIMOD bracket form) ---
SEQ=VSHGC[UNIMOD:4]VR
SEQ=SFSHQAGASSK
SEQ=IGHTVER


## Run prediction — pretrained checkpoint

`--evaluation` mode (vs the default `--denovo`) compares predictions
against the SEQ= labels in the annotated MGF and prints peptide / AA
precision at the end of the run. Output CSV contains the per-spectrum
predictions either way.


In [4]:
!instanovo transformer predict \
    --evaluation \
    --data-path  /content/data_mgf_annotated/ecoli/Ecoli_EV_2.instanovo.annotated.mgf \
    --output-path /content/result_finetune/instanovo/Ecoli_EV_2.pretrained.csv \
    --instanovo-model /content/bin/instanovo/instanovo-v1.2.0.ckpt \
    num_workers=4 \
    batch_size=512


[05/02/26 02:24:03] INFO     Initializing InstaNovo inference.                                                                                                                 
[05/02/26 02:24:05] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 02:24:07.818680: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 02:24:07.889387: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 02:24

## Run prediction — fine-tuned checkpoint

Same MGF and flags, just swap the model path to `model_best.ckpt`.


In [5]:
!instanovo transformer predict \
    --evaluation \
    --data-path  /content/data_mgf_annotated/ecoli/Ecoli_EV_2.instanovo.annotated.mgf \
    --output-path /content/result_finetune/instanovo/Ecoli_EV_2.finetuned.csv \
    --instanovo-model /content/model_finetune/instanovo/model_best.ckpt \
    num_workers=4 \
    batch_size=512


[05/02/26 02:25:27] INFO     Initializing InstaNovo inference.                                                                                                                 
[05/02/26 02:25:29] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 02:25:32.124130: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 02:25:32.191943: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 02:25

## Compare predictions to SEQ= labels (sanity check)

InstaNovo's `--evaluation` mode prints peptide / AA precision at the end
of each run above. This cell duplicates the peptide-level number locally
as a sanity check, joining each model's `predictions` column to the SEQ=
labels in the MGF on `scan_number` (the 0-based file index InstaNovo emits).

**Two caveats** (same as Casanovo earlier):

1. **Strict string match counts I/L disagreements as wrong** even though
   mass spec can't distinguish them. The real biological accuracy is a few
   points higher. The *delta* between pretrained and fine-tuned is still
   the right comparison.
2. **This is one number, not the full picture.** Downstream FDR-yield
   comparisons via the prediction CSVs we just wrote (Jetson-side
   pipeline) are the wastewater answer.


In [6]:
import pandas as pd

MGF = '/content/data_mgf_annotated/ecoli/Ecoli_EV_2.instanovo.annotated.mgf'

def load_seqs_from_mgf(mgf_path):
    """Return {scan_number (0-based file index): SEQ string} for every spectrum."""
    seqs = {}
    idx = -1
    with open(mgf_path) as f:
        for line in f:
            if line.startswith('BEGIN IONS'):
                idx += 1
            elif line.startswith('SEQ='):
                seqs[idx] = line.split('=', 1)[1].strip()
    return seqs

def compare(pred_csv, label, seqs):
    df = pd.read_csv(pred_csv)
    df['gt'] = df['scan_number'].map(seqs)
    labelled = df.dropna(subset=['gt'])
    exact = (labelled['predictions'] == labelled['gt']).sum()
    rate = exact / len(labelled) if len(labelled) else 0
    print(f'{label:<14} {len(df):>5} predictions   '
          f'{len(labelled):>5} labelled   '
          f'{exact:>5} exact   '
          f'{rate:>6.2%}')
    return labelled

seqs = load_seqs_from_mgf(MGF)
print(f'Ground-truth SEQ= labels in MGF: {len(seqs)}\n')
print(f'{"":<14} {"preds":>5}              {"in GT":>5}     {"exact":>5}     rate')
print('-' * 70)
pre  = compare('/content/result_finetune/instanovo/Ecoli_EV_2.pretrained.csv', 'pretrained',  seqs)
ft   = compare('/content/result_finetune/instanovo/Ecoli_EV_2.finetuned.csv',  'finetuned',   seqs)

# Side-by-side spot check for the first 8 GT scans
print('\nSide-by-side sample (first 8 GT-labelled scans):')
common = set(pre['scan_number']) & set(ft['scan_number'])
sample_scans = sorted(common)[:8]
pre_idx = pre.set_index('scan_number')
ft_idx  = ft.set_index('scan_number')
print(f'{"scan":>5}  {"GT":<32}  {"pretrained":<32}  {"finetuned":<32}')
print('-' * 110)
for s in sample_scans:
    gt   = pre_idx.loc[s, 'gt'][:32]
    p    = pre_idx.loc[s, 'predictions'][:32]
    f    = ft_idx.loc[s, 'predictions'][:32]
    print(f'{s:>5}  {gt:<32}  {p:<32}  {f:<32}')


Ground-truth SEQ= labels in MGF: 1273

               preds              in GT     exact     rate
----------------------------------------------------------------------
pretrained      1273 predictions    1273 labelled     633 exact   49.73%
finetuned       1273 predictions    1273 labelled     859 exact   67.48%

Side-by-side sample (first 8 GT-labelled scans):
 scan  GT                                pretrained                        finetuned                       
--------------------------------------------------------------------------------------------------------------
    0  VSHGC[UNIMOD:4]VR                 VSHGC[UNIMOD:4]VR                 VSHGC[UNIMOD:4]VR               
    1  SFSHQAGASSK                       SFSHQAGASSK                       SFSHQAGASSK                     
    2  IGHTVER                           LGHTVER                           LGHTVER                         
    3  IEQAPGQHGAR                       LEQAPGQHGAR                       IEQAPGQHGAR      

## Push prediction CSVs to Drive

Both prediction files go to `MyDrive/DL-Project/result_finetune/instanovo/`
for the Jetson-side pipeline (convert → check_alignment → NovoBoard FDR)
to consume.


In [7]:
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune/instanovo
!cp /content/result_finetune/instanovo/Ecoli_EV_2.pretrained.csv /content/drive/MyDrive/DL-Project/result_finetune/instanovo/
!cp /content/result_finetune/instanovo/Ecoli_EV_2.finetuned.csv  /content/drive/MyDrive/DL-Project/result_finetune/instanovo/
!ls -lh /content/drive/MyDrive/DL-Project/result_finetune/instanovo/


total 5.8M
-rw------- 1 root root 2.9M May  2 02:26 Ecoli_EV_2.finetuned.csv
-rw------- 1 root root 2.9M May  2 02:26 Ecoli_EV_2.pretrained.csv
